In [ ]:
!pip install ultralytics
!pip install pycocotools
!pip install tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.9 MB/s eta 0:00:00


In [ ]:
# 1️⃣ Montar Google Drive primero
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ====================================================================
# EVALUACIÓN YOLOv10-M — HerdNet EXACTO (AC_img y AC_final) TILES:1024
# ====================================================================

import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

# ------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# ------------------------------------------------------------
MODEL_PATH = "/content/drive/MyDrive/maia_yolov10_Exp1/tiles_1024_jpg_q100/weights/best.pt"
GT_JSON    = "/content/drive/MyDrive/general_dataset/groundtruth/json/big_size/test_big_size_A_B_E_K_WH_WB.json"
IMAGES_DIR = "/content/drive/MyDrive/general_dataset/test"

TILE   = 1024
OVERLAP = 0.25
STRIDE = int(TILE * (1 - OVERLAP))
CONF_THR = 0.25
IOU_THR  = 0.5


# ------------------------------------------------------------
# 1. CARGAR MODELO
# ------------------------------------------------------------
print("\nCargando modelo YOLOv10-M...")
model = YOLO(MODEL_PATH)
print("Modelo cargado correctamente ✔")


# ------------------------------------------------------------
# 2. GENERACIÓN DE TILES + NMS GLOBAL
# ------------------------------------------------------------

def generate_tiles(img):
    H, W, _ = img.shape
    tiles = []
    for y in range(0, H, STRIDE):
        for x in range(0, W, STRIDE):
            y2 = min(y + TILE, H)
            x2 = min(x + TILE, W)
            tiles.append((img[y:y2, x:x2], x, y))
    return tiles


def nms_global(dets, iou_thr=0.5):
    if len(dets) == 0:
        return []

    boxes = np.array([d["bbox"] for d in dets])
    scores = np.array([d["score"] for d in dets])

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    w  = boxes[:, 2]
    h  = boxes[:, 3]
    x2 = x1 + w
    y2 = y1 + h

    idxs = scores.argsort()[::-1]
    keep = []

    while len(idxs) > 0:
        i = idxs[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[idxs[1:]])
        yy1 = np.maximum(y1[i], y1[idxs[1:]])
        xx2 = np.minimum(x2[i], x2[idxs[1:]])
        yy2 = np.minimum(y2[i], y2[idxs[1:]])

        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        union = (w[i] * h[i]) + (w[idxs[1:]] * h[idxs[1:]]) - inter
        iou = inter / (union + 1e-6)

        idxs = idxs[1:][iou < iou_thr]

    return [dets[i] for i in keep]


# ------------------------------------------------------------
# 3. INFERENCIA SOBRE IMAGEN BIG (TILES)
# ------------------------------------------------------------
def infer_big_image(img_path, img_id):

    img = cv2.imread(img_path)
    H, W, _ = img.shape
    detections = []

    for tile, xo, yo in generate_tiles(img):

        res = model(tile, conf=CONF_THR, iou=IOU_THR, verbose=False)[0]

        if res.boxes is None:
            continue

        for b in res.boxes:
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()

            coco_id = int(b.cls[0].item()) + 1  # YOLO → COCO

            detections.append({
                "image_id": img_id,
                "category_id": coco_id,
                "bbox": [
                    float(x1 + xo),
                    float(y1 + yo),
                    float(x2 - x1),
                    float(y2 - y1)
                ],
                "score": float(b.conf[0].item())
            })

    return nms_global(detections, iou_thr=IOU_THR)


# ------------------------------------------------------------
# 4. INFERENCIA COMPLETA
# ------------------------------------------------------------
coco_gt = COCO(GT_JSON)
image_ids = coco_gt.getImgIds()

print(f"\nTotal imágenes a evaluar: {len(image_ids)}\n")

preds = []

for i, img_id in enumerate(tqdm(image_ids, desc="Inferencia BIG", ncols=100)):
    info = coco_gt.loadImgs(img_id)[0]
    path = os.path.join(IMAGES_DIR, info["file_name"])

    detections = infer_big_image(path, img_id)
    preds.extend(detections)


# ------------------------------------------------------------
# 5. GUARDAR PREDICCIONES
# ------------------------------------------------------------
PRED_JSON = "pred_yolov10m_tiles1024.json"
with open(PRED_JSON, "w") as f:
    json.dump(preds, f)

print(f"\nPredicciones guardadas en: {PRED_JSON}\n")


# ------------------------------------------------------------
# 6. COCO EVAL (AP50 / AP5095)
# ------------------------------------------------------------
coco_dt = coco_gt.loadRes(PRED_JSON)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

AP50_95 = coco_eval.stats[0]
AP50    = coco_eval.stats[1]


# ------------------------------------------------------------
# 7. GT por imagen + Pred por imagen (para AC_img)
# ------------------------------------------------------------
gt_counts = []
pred_counts = []

for img_id in image_ids:
    gt_ids = coco_gt.getAnnIds(imgIds=[img_id])
    gt_counts.append(len(gt_ids))

    pr = len([p for p in preds if p["image_id"] == img_id])
    pred_counts.append(pr)

gt_counts   = np.array(gt_counts)
pred_counts = np.array(pred_counts)


# ------------------------------------------------------------
# 8. MÉTRICAS HERDNET: MAE / RMSE / F1
# ------------------------------------------------------------
MAE  = np.mean(np.abs(gt_counts - pred_counts))
RMSE = np.sqrt(np.mean((gt_counts - pred_counts)**2))

TP = FP = FN = 0
for gt, pr in zip(gt_counts, pred_counts):
    if pr <= gt:
        TP += pr
        FN += gt - pr
    else:
        TP += gt
        FP += pr - gt

F1 = (2*TP) / (2*TP + FP + FN + 1e-9)


# ------------------------------------------------------------
# 9. AC_img y AC_final (FÓRMULA OFICIAL HERDNET)
# ------------------------------------------------------------
AC_img = []

for gt, pr in zip(gt_counts, pred_counts):
    ac = 1 - (abs(gt - pr) / max(gt, 1))  # fórmula oficial
    ac = max(0, ac)                       # clamp: no negativos
    AC_img.append(ac)

AC_FINAL = np.mean(AC_img) * 100


# ------------------------------------------------------------
# 10. MOSTRAR RESULTADOS
# ------------------------------------------------------------
print("\n================== RESULTADOS ==================")
print(f"AP50-95:        {AP50_95:.3f}")
print(f"AP50:           {AP50:.3f}")
print("-----------------------------------------------")
print(f"F1 HerdNet:     {F1:.3f}")
print(f"MAE:            {MAE:.3f}")
print(f"RMSE:           {RMSE:.3f}")
print(f"AC_final (%):   {AC_FINAL:.2f}")
print("================================================\n")



Cargando modelo YOLOv10-M...
Modelo cargado correctamente ✔
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!

Total imágenes a evaluar: 258



Inferencia BIG: 100%|█████████████████████████████████████████████| 258/258 [03:59<00:00,  1.08it/s]



Predicciones guardadas en: pred_yolov10m_tiles1024.json

Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.71s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.072
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.122
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.080
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.065
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.083
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.016
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.065
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.082
 Average 

In [18]:
# ====================================================================
# EVALUACIÓN YOLOv10-M — HerdNet EXACTO (AC_img y AC_final) TILES:960
# ====================================================================

import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

# ------------------------------------------------------------
# CONFIGURACIÓN GLOBAL (para modelo tiles 960x960)
# ------------------------------------------------------------
MODEL_PATH = "/content/drive/MyDrive/maia_yolo_train_960/yolov10m_960_tiles_v1/weights/best.pt"
GT_JSON    = "/content/drive/MyDrive/general_dataset/groundtruth/json/big_size/test_big_size_A_B_E_K_WH_WB.json"
IMAGES_DIR = "/content/drive/MyDrive/general_dataset/test"

TILE   = 960
OVERLAP = 0.25
STRIDE = int(TILE * (1 - OVERLAP))   # = 720
CONF_THR = 0.25
IOU_THR  = 0.5



# ------------------------------------------------------------
# 1. CARGAR MODELO
# ------------------------------------------------------------
print("\nCargando modelo YOLOv10-M...")
model = YOLO(MODEL_PATH)
print("Modelo cargado correctamente ✔")


# ------------------------------------------------------------
# 2. GENERACIÓN DE TILES + NMS GLOBAL
# ------------------------------------------------------------

def generate_tiles(img):
    H, W, _ = img.shape
    tiles = []
    for y in range(0, H, STRIDE):
        for x in range(0, W, STRIDE):
            y2 = min(y + TILE, H)
            x2 = min(x + TILE, W)
            tiles.append((img[y:y2, x:x2], x, y))
    return tiles


def nms_global(dets, iou_thr=0.5):
    if len(dets) == 0:
        return []

    boxes = np.array([d["bbox"] for d in dets])
    scores = np.array([d["score"] for d in dets])

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    w  = boxes[:, 2]
    h  = boxes[:, 3]
    x2 = x1 + w
    y2 = y1 + h

    idxs = scores.argsort()[::-1]
    keep = []

    while len(idxs) > 0:
        i = idxs[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[idxs[1:]])
        yy1 = np.maximum(y1[i], y1[idxs[1:]])
        xx2 = np.minimum(x2[i], x2[idxs[1:]])
        yy2 = np.minimum(y2[i], y2[idxs[1:]])

        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        union = (w[i] * h[i]) + (w[idxs[1:]] * h[idxs[1:]]) - inter
        iou = inter / (union + 1e-6)

        idxs = idxs[1:][iou < iou_thr]

    return [dets[i] for i in keep]


# ------------------------------------------------------------
# 3. INFERENCIA SOBRE IMAGEN BIG (TILES)
# ------------------------------------------------------------
def infer_big_image(img_path, img_id):

    img = cv2.imread(img_path)
    H, W, _ = img.shape
    detections = []

    for tile, xo, yo in generate_tiles(img):

        res = model(tile, conf=CONF_THR, iou=IOU_THR, verbose=False)[0]

        if res.boxes is None:
            continue

        for b in res.boxes:
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()

            coco_id = int(b.cls[0].item()) + 1  # YOLO → COCO

            detections.append({
                "image_id": img_id,
                "category_id": coco_id,
                "bbox": [
                    float(x1 + xo),
                    float(y1 + yo),
                    float(x2 - x1),
                    float(y2 - y1)
                ],
                "score": float(b.conf[0].item())
            })

    return nms_global(detections, iou_thr=IOU_THR)


# ------------------------------------------------------------
# 4. INFERENCIA COMPLETA
# ------------------------------------------------------------
coco_gt = COCO(GT_JSON)
image_ids = coco_gt.getImgIds()

print(f"\nTotal imágenes a evaluar: {len(image_ids)}\n")

preds = []

for i, img_id in enumerate(tqdm(image_ids, desc="Inferencia BIG", ncols=100)):
    info = coco_gt.loadImgs(img_id)[0]
    path = os.path.join(IMAGES_DIR, info["file_name"])

    detections = infer_big_image(path, img_id)
    preds.extend(detections)


# ------------------------------------------------------------
# 5. GUARDAR PREDICCIONES
# ------------------------------------------------------------
PRED_JSON = "pred_yolov10m_tiles1960.json"
with open(PRED_JSON, "w") as f:
    json.dump(preds, f)

print(f"\nPredicciones guardadas en: {PRED_JSON}\n")


# ------------------------------------------------------------
# 6. COCO EVAL (AP50 / AP5095)
# ------------------------------------------------------------
coco_dt = coco_gt.loadRes(PRED_JSON)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

AP50_95 = coco_eval.stats[0]
AP50    = coco_eval.stats[1]


# ------------------------------------------------------------
# 7. GT por imagen + Pred por imagen (para AC_img)
# ------------------------------------------------------------
gt_counts = []
pred_counts = []

for img_id in image_ids:
    gt_ids = coco_gt.getAnnIds(imgIds=[img_id])
    gt_counts.append(len(gt_ids))

    pr = len([p for p in preds if p["image_id"] == img_id])
    pred_counts.append(pr)

gt_counts   = np.array(gt_counts)
pred_counts = np.array(pred_counts)


# ------------------------------------------------------------
# 8. MÉTRICAS HERDNET: MAE / RMSE / F1
# ------------------------------------------------------------
MAE  = np.mean(np.abs(gt_counts - pred_counts))
RMSE = np.sqrt(np.mean((gt_counts - pred_counts)**2))

TP = FP = FN = 0
for gt, pr in zip(gt_counts, pred_counts):
    if pr <= gt:
        TP += pr
        FN += gt - pr
    else:
        TP += gt
        FP += pr - gt

F1 = (2*TP) / (2*TP + FP + FN + 1e-9)


# ------------------------------------------------------------
# 9. AC_img y AC_final (FÓRMULA OFICIAL HERDNET)
# ------------------------------------------------------------
AC_img = []

for gt, pr in zip(gt_counts, pred_counts):
    ac = 1 - (abs(gt - pr) / max(gt, 1))  # fórmula oficial
    ac = max(0, ac)                       # clamp: no negativos
    AC_img.append(ac)

AC_FINAL = np.mean(AC_img) * 100


# ------------------------------------------------------------
# 10. MOSTRAR RESULTADOS
# ------------------------------------------------------------
print("\n================== RESULTADOS TILES 960 ==================")
print(f"AP50-95:        {AP50_95:.3f}")
print(f"AP50:           {AP50:.3f}")
print("-----------------------------------------------")
print(f"F1 HerdNet:     {F1:.3f}")
print(f"MAE:            {MAE:.3f}")
print(f"RMSE:           {RMSE:.3f}")
print(f"AC_final (%):   {AC_FINAL:.2f}")
print("================================================\n")



Cargando modelo YOLOv10-M...
Modelo cargado correctamente ✔
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!

Total imágenes a evaluar: 258



Inferencia BIG: 100%|█████████████████████████████████████████████| 258/258 [04:18<00:00,  1.00s/it]



Predicciones guardadas en: pred_yolov10m_tiles1960.json

Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.79s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.067
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.118
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.068
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.055
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.080
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.016
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.061
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.078
 Average 